# 01. Data Preparation

First notebook of four. The raw Kaggle file is messy on purpose: currency
strings inside numeric columns, two date formats, duplicate ad ids and a few
rows where clicks exceed impressions. This notebook cleans it, engineers the
metrics the rest of the analysis depends on, and saves the processed file.

If you have not placed the Kaggle CSV in `data/raw/`, run
`python scripts/make_sample_data.py` first to generate a synthetic stand-in
with the same schema.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from src import data_prep, eda, stats_analysis, modeling
from src.plotting import apply_style

apply_style()
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

## Raw file inspection

A quick look before touching anything. Note the mixed types in `Cost` and
`Sale_Amount` and the two date formats in `Ad_Date`.

In [2]:
raw = data_prep.load_raw()
raw.sample(8, random_state=1)

,Ad_ID,Ad_Date,Campaign_Name,Keyword,Device,Location,Impressions,Clicks,Cost,Leads,Conversions,Conversion Rate,Sale_Amount
161,AD102190,2024-11-07,Search_Shoes_Bangalore,shoe sale,Desktop,Bangalore,480,28,518.77,6,3.0000,10.7100,11582.41
230,AD103023,15-05-2024,Search_Shoes_Pune,buy from us stride,Mobile,Pune,982,57,456.11,5,4.0000,7.0200,10024.02
366,AD102513,2024-12-15,Search_Shoes_Bangalore,womens walking shoes,Desktop,Bangalore,1158,53,661.78,6,NaN,7.5500,8366.16
872,AD103411,2024-06-11,Search_Shoes_Hyderabad,canvas shoes,Mobile,Hyderabad,1874,60,797.74,6,4.0000,6.6700,9180.8
3187,AD102121,2024-09-07,Search_Shoes_Hyderabad,running shoes,Mobile,Hyderabad,1809,113,1978.2,5,5.0000,4.4200,15800.3
2448,AD102458,2024-06-22,Search_Shoes_Mumbai,best trail running shoes,Mobile,Mumbai,761,42,437.01,5,3.0000,7.1400,7918.62
2894,AD100567,2024-03-08,Search_Shoes_Mumbai,formal shoes for men,Desktop,Mumbai,394,20,417.52,2,1.0000,5.0000,3474.45
1658,AD101991,2024-05-20,Search_Shoes_Delhi,best trail running shoes,Desktop,Delhi,381,6,89.57,1,1.0000,16.6700,1882.58


In [3]:
raw.dtypes

Ad_ID                  str
Ad_Date                str
Campaign_Name          str
Keyword                str
Device                 str
Location               str
Impressions          int64
Clicks               int64
Cost                   str
Leads                int64
Conversions        float64
Conversion Rate    float64
Sale_Amount            str
dtype: object

## Cleaning

`data_prep.clean` normalizes column names, strips currency symbols, parses
both date formats, drops duplicates and removes rows that fail physical
sanity checks (clicks above impressions, conversions above clicks).

In [4]:
tidy = data_prep.clean(raw)
print(f"Raw rows: {len(raw)}")
print(f"Clean rows: {len(tidy)}")
tidy.dtypes

Raw rows: 3535
Clean rows: 3500


ad_id                         str
ad_date            datetime64[us]
campaign_name                 str
keyword                       str
device                        str
location                      str
impressions               float64
clicks                    float64
cost                      float64
leads                     float64
conversions               float64
conversion_rate           float64
sale_amount               float64
dtype: object

## Feature engineering

The derived metrics the project is actually about:

- `ctr` clicks / impressions and `cvr` conversions / clicks
- `cpc`, `cpa`, `roas`, `revenue_per_click`
- `keyword_length`, `keyword_word_count`, and `is_branded`, flagged when the
  query contains brand tokens
- calendar features: `month`, `day_of_week`, `is_weekend`

In [5]:
df = data_prep.engineer(tidy)
df.to_csv("data/processed/cleaned_google_ads.csv", index=False)
df[["keyword", "device", "location", "ctr", "cvr", "cpc",
    "is_branded", "keyword_word_count"]].head(8)

,keyword,device,location,ctr,cvr,cpc,is_branded,keyword_word_count
0,Tennis Shoes,Mobile,Delhi,0.0438,0.0294,11.4926,0,2
1,White Sneakers Under 2000,Desktop,Hyderabad,0.0342,0.1538,13.1258,0,4
2,Orthopedic Walking Shoes,Mobile,Bangalore,0.0575,0.0952,14.4462,0,3
3,Stride Official Store,Mobile,Bangalore,0.0798,0.1014,11.3895,1,3
4,Tennis Shoes,Desktop,Mumbai,0.0495,0.0822,9.7012,0,2
5,Cheap Sports Shoes,Mobile,Bangalore,0.0667,0.0286,8.7913,0,3
6,Stride Shop Online,Desktop,Hyderabad,0.0210,0.0909,5.2627,1,3
7,Shoe Store Near Me,Tablet,Chennai,0.0284,0.1053,14.0061,1,4


In [6]:
print(f"Date range: {df['ad_date'].min().date()} to {df['ad_date'].max().date()}")
print(f"Branded share of ads: {df['is_branded'].mean():.1%}")
print(f"Missing CVR (zero-click ads): {df['cvr'].isna().mean():.1%}")

Date range: 2024-01-01 to 2024-12-30
Branded share of ads: 21.3%
Missing CVR (zero-click ads): 2.1%


The processed file is saved to `data/processed/cleaned_google_ads.csv`.
Continue with `02_exploratory_analysis.ipynb`.